In [1]:
!pip install transformers torch

In [2]:
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration

In [3]:
import pandas as pd

In [4]:
df=pd.read_csv('final_matched_dataset (1).csv')

In [5]:
df['ingredients'] = df['ingredients'].str.lower()
df['ingredients'] = df['ingredients'].str.replace("...", "")

In [6]:
df['text'] = df['name'].fillna('').astype(str) + " " + \
             df['ingredients'].fillna('').astype(str) + " " + \
             df['category'].fillna('').astype(str)

In [7]:
df['target'] = (
    "calories: " + df['calories'].astype(str) + " " +
    "protein: " + df['protein'].astype(str) + " " +
    "fat: " + df['fat'].astype(str) + " " +
    "carb: " + df['carb'].astype(str)
)

In [8]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = "t5-small"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [9]:
def preprocess_function(examples):
    inputs = ["predict nutrients: " + x for x in examples["text"]]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding=True)

    labels = tokenizer(examples["target"], max_length=64, truncation=True, padding=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [10]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2)

In [11]:
from transformers import Trainer, TrainingArguments


In [12]:
from transformers import TrainingArguments, IntervalStrategy

training_args = TrainingArguments(
    output_dir="./t5_model",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy=IntervalStrategy.EPOCH,
    save_strategy=IntervalStrategy.EPOCH,
)

In [13]:
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq

train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(test_df)

train_dataset = train_dataset.map(preprocess_function, batched=True)
eval_dataset = eval_dataset.map(preprocess_function, batched=True)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

Map:   0%|          | 0/11063 [00:00<?, ? examples/s]

Map:   0%|          | 0/2766 [00:00<?, ? examples/s]

In [14]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.387055,1.261989
2,1.306633,1.207950
3,1.267835,1.184132


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4149, training_loss=1.3805185806667928, metrics={'train_runtime': 509.6819, 'train_samples_per_second': 65.117, 'train_steps_per_second': 8.14, 'total_flos': 1122964762263552.0, 'train_loss': 1.3805185806667928, 'epoch': 3.0})

In [15]:
def predict(text):
    input_text = "predict nutrients: " + text

    inputs = tokenizer(input_text, return_tensors="pt")

    output = model.generate(**inputs, max_length=64)

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [16]:
def predict(text):
    input_text = "predict nutrients: " + text

    inputs = tokenizer(input_text, return_tensors="pt")

    output = model.generate(**inputs, max_length=64)

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [17]:
predict("chicken onion garlic rice")

RuntimeError: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)